# Video

Demonstrating different responses of the retinal layers to different types of motion - lateral and approaching. The lateral motion generates more spikes in the ganglion layer than does the lateral motion.

In [1]:
from typing import *

# --------------------------------------
import torch as pt

# --------------------------------------
import numpy as np

# --------------------------------------
from functools import partial

# --------------------------------------
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
plt.ioff()

# --------------------------------------
import cv2 as cv

# --------------------------------------
from pathlib import Path

# --------------------------------------
from IPython.display import Video

# --------------------------------------
from pyrception import conf
import pyrception.visual.util.types as pct
import pyrception.util.functions as pf
from pyrception.visual.layers import ReceptorLayer
from pyrception.visual.layers import HorizontalLayer
from pyrception.visual.layers import BipolarLayer
from pyrception.visual.layers import AmacrineLayer
from pyrception.visual.layers import GanglionLayer
from pyrception.visual.util.types import ImagePlot
from pyrception.visual.util.types import ScatterPlot

Pyrception | 2024-06-27@14:03:03 | Logger configured.
Pyrception | 2024-06-27@14:03:03 | Default PyTorch tensor type: 'torch.float32'.
Pyrception | 2024-06-27@14:03:03 | Default PyTorch device: 'cpu'.


## Open and process the video files

In [2]:
def frame_generator(
    path: Path,
    max_frames: int = None,
    scale: float = None,
):

    cap = cv.VideoCapture(path)
    frame_idx = 0

    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))

    new_size = None
    if scale is not None:
        if not (0.0 < scale <= 1.0):
            raise ValueError(
                f"Invalid scale {scale} (must be a positive number in (0.0, 1.0])."
            )
        new_size = (int(width * scale), int(height * scale))

    while True:
        # Read a frame
        (processing, frame) = cap.read()

        if (
            processing is None
            or frame is None
            or (max_frames is not None and frame_idx >= max_frames)
        ):
            print("\n==[ Exiting...")
            break

        frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        if new_size is not None:
            frame = cv.resize(frame, new_size)

        frame_idx += 1

        print(f"==[ Processing frame {frame_idx:>4d} (size: {new_size})...\r", end="")
        yield frame.astype(np.float32)

In [3]:
lateral_motion_video_path = Path("./resources/lateral_motion.mp4")
approaching_motion_video_path = Path("./resources/approaching_motion.mp4")
if not lateral_motion_video_path.exists():
    raise FileNotFoundError(
        f"Invalid path for the lateral motion video: {lateral_motion_video_path}"
    )
if not approaching_motion_video_path.exists():
    raise FileNotFoundError(f"Invalid path for the approaching motion video: {approaching_motion_video_path}")

lateral_motion_frames = pt.from_numpy(np.stack([f for f in frame_generator(lateral_motion_video_path, scale=0.5)], axis=0)).to(
    conf.device
)
approaching_motion_frames = pt.from_numpy(np.stack([f for f in frame_generator(approaching_motion_video_path, scale=0.5)], axis=0)).to(
    conf.device
)

==[ Processing frame  120 (size: (640, 360))...
==[ Exiting...
==[ Processing frame  120 (size: (640, 360))...
==[ Exiting...


In [4]:
(h, w) = lateral_motion_frames[0].shape
size = (h, w)

plot_kw = {}

image_kw = {}

scatter_kw = {
    "axis": True,
    "spines": True,
}

In [5]:
(fig, _, _) = pf.plot(lateral_motion_frames[0], **plot_kw)

In [6]:
(fig, _, _) = pf.plot(approaching_motion_frames[0], **plot_kw)

In [7]:
def implot(x: np.ndarray, **kwargs):
    kw = image_kw.copy()
    kw.update(kwargs)
    return ImagePlot(x, **kwargs)


def scatplot(x: np.ndarray, **kwargs):
    kw = image_kw.copy()
    kw.update(kwargs)
    return ScatterPlot(x, **kwargs)

## Receptor layer

In [8]:
rl_params = {
    "size": size,
}

rl = ReceptorLayer(**rl_params)

Pyrception | 2024-06-27@14:03:04 | Receptor | Initialising...
Pyrception | 2024-06-27@14:03:04 | Receptor | Initialised.


## Horizontal layer

In [9]:
hl_params = {
    "size": size,
    "receptor": rl,
    "sectors": 64,
    "rf_params": {
        "kernel_params": {
            "filter": pct.KernelFilter.Gaussian,
            "min_size": 1,
            "scale": 1.0,
        }
    },
}

hl = HorizontalLayer(**hl_params)

Pyrception | 2024-06-27@14:03:04 | Horizontal | Initialising...
Pyrception | 2024-06-27@14:03:04 | Horizontal | Horizontal RFs | Creating receptive fields...


Neurons created: 100%|██████████| 2515/2515 [00:04<00:00, 515.79it/s]
/home/hobbes/code/gitlab/pyrception/remote/pyrception/visual/rf.py:908: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /opt/conda/conda-bld/pytorch_1716905969073/work/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  .to_sparse_csr()


Pyrception | 2024-06-27@14:03:10 | Horizontal | Horizontal RFs | Receptive fields created for 2515 neurons.
Pyrception | 2024-06-27@14:03:10 | Horizontal | Initialised.


## Bipolar layer

In [10]:
bl_params = {
    "size": size,
    "receptor": rl,
    "horizontal": hl,
    "sectors": 128,
    "forgetting_range": (0.45, 0.5),
    "rf_params": {
        "kernel_params": {
            "filter": pct.KernelFilter.Flat,
        }
    },
}

bl = BipolarLayer(**bl_params)

Pyrception | 2024-06-27@14:03:10 | Bipolar | Initialising...
Pyrception | 2024-06-27@14:03:10 | Bipolar | Bipolar RFs | Creating receptive fields...


Neurons created: 100%|██████████| 7995/7995 [00:11<00:00, 716.67it/s]


Pyrception | 2024-06-27@14:03:22 | Bipolar | Bipolar RFs | Receptive fields created for 7995 neurons.
Pyrception | 2024-06-27@14:03:22 | Bipolar | Forgetting rate range: 0.450 - 0.500
Pyrception | 2024-06-27@14:03:22 | Bipolar | Initialised.


## Amacrine layer

In [11]:
al_params = {
    "size": size,
    "bipolar": bl,
    "sectors": 96,
    "rf_params": {
        "kernel_params": {
            "filter": pct.KernelFilter.Flat,
            "min_size": 3,
        }
    },
}

al = AmacrineLayer(**al_params)

Pyrception | 2024-06-27@14:03:22 | Amacrine | Initialising...
Pyrception | 2024-06-27@14:03:22 | Amacrine | Amacrine RFs | Creating receptive fields...


Neurons created: 100%|██████████| 4985/4985 [00:01<00:00, 2732.43it/s]


Pyrception | 2024-06-27@14:03:24 | Amacrine | Amacrine RFs | Receptive fields created for 4985 neurons.
Pyrception | 2024-06-27@14:03:24 | Amacrine | Initialised.


## Ganglion layer

In [30]:
gl_bl_params = {
    "kernel_params": {
        "scale": 1.0,
        "filter": pct.KernelFilter.Flat,
    }
}
gl_al_params = {
    "kernel_params": {
        "scale": 2.0,
        "filter": pct.KernelFilter.Flat,
    },
}


gl_params = {
    "size": size,
    "bipolar": bl,
    "amacrine": al,
    "sectors": 64,
    "bipolar_params": gl_bl_params,
    "amacrine_params": gl_al_params,
    "inhibition_scale": 1.5,
}


class ApproachDetectionGanglionLayer(GanglionLayer):
    def forward(self):
        """
        Implements a looming detector for approaching bright objects.
        """

        # ON centre / OFF surround

        bl_on = self.convolve(
            self.bipolar_rfs.rfs,
            self.bipolar.off,
        )

        al = self.inhibition_scale * self.convolve(
            self.amacrine_rfs.rfs,
            self.amacrine.off,
        )
        spikes = pt.where(-0.05 + bl_on - al >= 0, 1, 0)

        return spikes


gl = ApproachDetectionGanglionLayer(**gl_params)

Pyrception | 2024-06-27@14:10:30 | Ganglion | Initialising...
Pyrception | 2024-06-27@14:10:30 | Ganglion | Bipolar RFs | Creating receptive fields...


Neurons created: 100%|██████████| 2515/2515 [00:00<00:00, 3790.30it/s]

Pyrception | 2024-06-27@14:10:31 | Ganglion | Bipolar RFs | Receptive fields created for 2515 neurons.
Pyrception | 2024-06-27@14:10:31 | Ganglion | Amacrine RFs | Creating receptive fields...



Neurons created: 100%|██████████| 2515/2515 [00:00<00:00, 3023.07it/s]

Pyrception | 2024-06-27@14:10:32 | Ganglion | Amacrine RFs | Receptive fields created for 2515 neurons.
Pyrception | 2024-06-27@14:10:32 | Ganglion | Initialised.


Pass a single frame through all the layers and plot the results into the designated slots.

In [49]:
@conf.logger.catch
def process(
    frame: pt.Tensor,
    axes: plt.Axes,
    mappables: List[PathCollection],
):

    # Blank canvas
    canvas = np.zeros((h, w))

    # Receptors
    # ==================================================
    rl.forward(frame)

    # Plot the raw input
    mappables[0].set_data(frame.numpy())
    mappables[0].set_clim(frame.min(), frame.max())

    # Horizontal cells
    # ==================================================
    (hl_activation, hl_feedback) = hl.forward()
    hl_feedback = hl_feedback.reshape(hl.rfs.height, hl.rfs.width)

    # Plot the activation of the horizontal cells
    hl_activation_canvas = canvas.copy()
    hl_activation_canvas[hl.rfs.cell_coordinates[:, 0], hl.rfs.cell_coordinates[:, 1]] = hl_activation.numpy()
    mappables[1].set_data(hl_activation_canvas)
    mappables[1].set_clim(hl_activation_canvas.min(), hl_activation_canvas.max())

    # Plot the feedback signal (mean illumination map)
    mappables[2].set_data(hl_feedback.numpy())
    mappables[2].set_clim(hl_feedback.min(), hl_feedback.max())

    # Plot the scaled input
    scaled = frame - hl_feedback
    mappables[3].set_data(scaled.numpy())
    mappables[3].set_clim(scaled.min(), scaled.max())

    # Bipolar cells
    # ==================================================
    (bl_on, bl_off, scaled, bp_activation) = bl.forward()

    # Plot the activation of ON and OFF bipolar cells
    bl_on_canvas = canvas.copy()
    bl_off_canvas = canvas.copy()

    bl_on_canvas[bl.rfs.cell_coordinates[:, 0], bl.rfs.cell_coordinates[:, 1]] = bl_on.numpy()
    bl_off_canvas[bl.rfs.cell_coordinates[:, 0], bl.rfs.cell_coordinates[:, 1]] = bl_off.numpy()

    mappables[4].set_data(bl_on_canvas)
    mappables[4].set_clim(bl_on_canvas.min(), bl_on_canvas.max())

    mappables[5].set_data(bl_off_canvas)
    mappables[5].set_clim(bl_off_canvas.min(), bl_off_canvas.max())

    # Amacrine cells
    # ==================================================
    (al_on, al_off) = al.forward()

    # Plot the activation of ON and OFF amarcine cells
    al_on_canvas = canvas.copy()
    al_off_canvas = canvas.copy()

    al_on_canvas[al.rfs.cell_coordinates[:, 0], al.rfs.cell_coordinates[:, 1]] = al_on.numpy()
    al_off_canvas[al.rfs.cell_coordinates[:, 0], al.rfs.cell_coordinates[:, 1]] = al_off.numpy()

    mappables[6].set_data(al_on_canvas)
    mappables[6].set_clim(al_on_canvas.min(), al_on_canvas.max())

    mappables[7].set_data(al_off_canvas)
    mappables[7].set_clim(al_off_canvas.min(), al_off_canvas.max())

    # Ganglion cells
    # ==================================================
    gl_spikes = gl.forward()

    # Plot the activation of ON-centre and OFF-centre ganglion cells
    gl_canvas = canvas.copy()

    gl_canvas[gl.bipolar_rfs.cell_coordinates[:, 0], gl.bipolar_rfs.cell_coordinates[:, 1]] = gl_spikes.numpy()

    mappables[8].set_data(gl_canvas)
    mappables[8].set_clim(0, 1)


In [50]:
def animator(
    frame: pt.Tensor,
    axes: plt.Axes,
    mappables: List[PathCollection],
):

    process(frame, axes, mappables)
    return mappables

A simple frame iterator - used by the animation routine.

In [51]:
def frame_iterator(frames: pt.Tensor):
    for frame_idx, frame in enumerate(frames):

        print(f"==[ Processing frame {frame_idx:>5d}...\r", end="")
        yield frame

Make a plot grid for the input and all layers

In [52]:
def make_plot_grid(title: str = "Retinomorphic processing pipeline"):

    init_frame = pt.zeros((h, w))

    (fig, axes, mappables) = pf.plot(
        [
            [
                implot(init_frame, title="Raw input"),
                implot(init_frame, title="Horizontal cell activity"),
                implot(
                    init_frame, title="Horizontal cell feedback (mean illumination map)"
                ),
            ],
            [
                implot(init_frame, title="Scaled raw input"),
                implot(init_frame, title="ON bipolar"),
                implot(init_frame, title="OFF bipolar"),
            ],
            [
                implot(init_frame, title="ON-driven amacrine"),
                implot(init_frame, title="OFF-driven amacrine"),
                implot(init_frame, title="Ganglion"),
            ],
        ],
        figsize=(16, 8),
        title=title,
    )

    return (fig, axes, mappables)

In [53]:
fps = 25
video_format = "mp4"

In [54]:
# Animation - lateral motion
# ==================================================
(fig_lat, axes_lat, mappables_lat) = make_plot_grid()
_, filename_lateral_motion = pf.animate(
    fig_lat,
    partial(animator, axes=axes_lat, mappables=mappables_lat),
    partial(frame_iterator, lateral_motion_frames),
    title="Retina response - lateral motion",
    fps=fps,
    output_dir="./output/lateral_motion",
    format=video_format,
)

In [62]:
Video(filename_lateral_motion, width=750)

In [56]:
# Animation - approaching motion
# ==================================================
(fig_app, axes_app, mappables_app) = make_plot_grid()
ani, filename_approaching_motion = pf.animate(
    fig_app,
    partial(animator, axes=axes_app, mappables=mappables_app),
    partial(frame_iterator, approaching_motion_frames),
    title="Retina response - approaching motion",
    fps=fps,
    output_dir="./output/approaching_motion",
    format=video_format,
)

In [63]:
Video(filename_approaching_motion, width=750)